## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [12]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [13]:
MY_BUBBLE_FILE = "pro_european.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: pro_european
Texte: 50


,id,agent,text
0,yt_FAe_Sqr2vfU_UgzfkQjlhFS0o3ZaCMl4AaABAg,Pro-european,Mă bucur să văd un Președinte care se comportă...
1,yt_Gk7qe_F1KWE_Ugw6Diry83RDiew6P4V4AaABAg,Pro-european,Ce sunt cu torți trolli ăștia din comentarii? ...
2,yt_Pk7Qkhnt4Bs_UgwxY19uJCh0c9RP4U54AaABAg,Pro-european,Sanatate si putere de munca domnule presedinte...
3,yt_6_Hc2S02Duw_UgwTw7_YpNZEUGkYDb94AaABAg,Pro-european,"Nu are Ce cauta pe teritoriulRomaniei, indifer..."
4,yt_yEuctxNb4O0_UgxTCkwcP96Sb5_Rpht4AaABAg,Pro-european,Tipul care și-a dat demisia în noiembrie la cu...


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [14]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Mă bucur să văd un Președinte care se comportă firesc și zâmbește atât de frumos. Îmi dă speranța că lucrurile merg pe făgașul normalității. 👏🏻


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [15]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

Batches: 100%|██████████| 2/2 [00:00<00:00,  3.33it/s]

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [16]:
# TODO student:
# Bula mea are 50 texte.
# Au fost generați 50 vectori.
# A doua valoare din embeddings.shape reprezintă dimensiunea vectorului.

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [17]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\pro_european
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [18]:
# TODO student:
# index.faiss există: ...
# index.pkl există: ...
# index.ntotal este egal cu numărul de texte: ...

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În următorul continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [19]:
# Text nou introdus în aplicație

input_text = "Bolojan a fost la Bruxelles și a adus bani pentru Oradea. E un primar care se ocupă de orașul lui, nu stă să comenteze pe Facebook ce face alții."

In [20]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [21]:
# query_vector

In [22]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.561
Text: de apreciat tot ce ati realizat pana acum in Oradea sunteti demn de toata lauda felicitari !!!Oradea un oras european !!!

Rezultat 2
Scor: 0.531
Text: Hai ca se poate sa avem un Bucuresti demn de o capitala europeană! Ana la primarie! 💚💚💚

Rezultat 3
Scor: 0.392
Text: Aici in Italia este o firma care se ocupa cu parcarea, amenzile se platesc la politie, politia este necesara, nu pt parcare, pt linistea si ocrotirea odinii publice si legile trebuie executate.

Rezultat 4
Scor: 0.384
Text: În primul rând, când vine vorba de taxe, uită-te la ce accize se plătesc la combustibil. Deci, da, unu care merge cu mașina plătește mai multe taxe decât un pieton. Însă, în București e o nenorocire făcută de toți șoferii. Chiar dacă îl costă 10 lei el nu plătește ci pune mașina ca un rahat. Prima manevră este mărirea taxelor la modul cel mai mic 200 de euro și de la 2000 de euro dacă are mai mult de 10 ani. Să pună taxa de oraș pentru cine vine din afara lui. Amenzi și r

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.